# 04 — Build weekly prices
 **Purpose:** one row per item, state, district, premise type and Monday-start week.
 **Saved:** only `weekly_prices.parquet`, excluding incomplete calendar weeks at the data-scope boundaries.
 The median uses all recorded daily prices; frequently monitored premises contribute more records. It is not a fixed-premise price index. Seven monitoring days are not required.


In [1]:
import pandas as pd
import json
import hashlib
import pricecatcher_config as cfg
from importlib import reload
cfg = reload(cfg)
print("Experiment:", cfg.EXPERIMENT, "| Training cutoff:", cfg.TRAIN_END_DATE)

cleaning = json.loads((cfg.STAGE3_DIR / "cleaning_summary.json").read_text())
preparation = cleaning["preparation"]
expected = {**cfg.PREPARATION_SETTINGS, "source_manifest_sha256": hashlib.sha256(
    (cfg.STAGE1_DIR / "source_manifest.json").read_bytes()).hexdigest()}
if preparation != expected:
    raise ValueError("Preparation settings/source changed. Rerun 02–05.")
df = pd.read_parquet(cfg.STAGE3_MODEL_READY_PARQUET)
df["date"] = pd.to_datetime(df["date"])
df["week_start"] = df["date"] - pd.to_timedelta(df["date"].dt.weekday, unit="D")
print("Daily records:", len(df))


Experiment: holdout | Training cutoff: 2025-12-31
Daily records: 4722617


## 1. Aggregate prices and supporting evidence
 Keep median/min/max/standard deviation, record counts, premise counts, monitoring days and suspicious-record counts. Standard deviation for a single record stays missing.


In [2]:
weekly = df.groupby(cfg.GROUP_KEYS + ["week_start"]).agg(
    weekly_median_price=("price", "median"),
    weekly_min_price=("price", "min"),
    weekly_max_price=("price", "max"),
    weekly_std_price=("price", "std"),
    record_count=("price", "size"),
    premise_count=("premise_code", "nunique"),
    observed_days=("date", "nunique"),
    suspicious_record_count=("is_suspicious_price", "sum")
).reset_index().sort_values(cfg.GROUP_KEYS + ["week_start"])

# Equal-premise comparator: one median per shop, then a median across shops.
# This reduces unequal monitoring weights; changing shop identities can still matter.
premise_week = df.groupby(cfg.GROUP_KEYS + ["week_start", "premise_code"])["price"].median()
equal_premise = premise_week.groupby(cfg.GROUP_KEYS + ["week_start"]).median().rename("equal_premise_median")
weekly = weekly.merge(equal_premise.reset_index(), on=cfg.GROUP_KEYS + ["week_start"],
                      how="left", validate="one_to_one")
del premise_week, equal_premise

# Keep standard deviation missing for n=1: it is undefined, not zero.
weekly["single_record_week"] = weekly["record_count"] == 1
weekly["week_end"] = weekly["week_start"] + pd.Timedelta(days=6)
scope_start = pd.Period(cfg.START_MONTH, freq="M").start_time.normalize()
scope_end = pd.Timestamp(cfg.EVALUATION_END_DATE)
weekly["partial_scope_week"] = (weekly["week_start"] < scope_start) | (weekly["week_end"] > scope_end)
# This flags scope boundaries, not whether monitoring occurred every day.
weekly["gap_days_from_previous"] = weekly.groupby(cfg.GROUP_KEYS)["week_start"].diff().dt.days
if weekly.duplicated(cfg.GROUP_KEYS + ["week_start"]).any():
    raise ValueError("Weekly keys must be unique.")
if not weekly["weekly_median_price"].gt(0).all():
    raise ValueError("Weekly median prices must be positive.")
print("Weekly observations:", len(weekly))


Weekly observations: 494832


## 2. Check gaps and save complete-scope weeks
 Missing weeks remain missing. Boundary exclusion only removes incomplete first/last calendar weeks; it does not remove local gaps or guarantee good coverage.


In [3]:
coverage = weekly.groupby(cfg.GROUP_KEYS).agg(
    weeks_present=("week_start", "size"), first_week=("week_start", "min"), last_week=("week_start", "max"))
coverage["missing_weeks_in_span"] = ((coverage["last_week"] - coverage["first_week"]).dt.days // 7 + 1) - coverage["weeks_present"]
print("Local series:", len(coverage))
print("Series with fewer than 10 observed weeks:", int((coverage["weeks_present"] < 10).sum()))
print("Gaps longer than one week:", int((weekly["gap_days_from_previous"] > 7).sum()))
print("Single-record weeks:", int(weekly["single_record_week"].sum()))
print("Boundary rows excluded:", int(weekly["partial_scope_week"].sum()))
display(coverage.head(5))

complete = weekly.loc[~weekly["partial_scope_week"]].drop(columns=["partial_scope_week", "single_record_week"])
if complete.empty:
    raise ValueError("No complete-scope weekly observations.")
complete.to_parquet(cfg.STAGE4_WEEKLY_PARQUET, index=False)
print("Saved complete-scope weekly rows:", len(complete))
print("Next: Notebook 05.")

(cfg.STAGE4_DIR / "preparation_settings.json").write_text(json.dumps(preparation, indent=2))


Local series: 9625
Series with fewer than 10 observed weeks: 1163
Gaps longer than one week: 53194
Single-record weeks: 99783
Boundary rows excluded: 13049


weeks_present  \
project_group item_code item                   unit state district   premise_type                              
Chicken       1         AYAM BERSIH - STANDARD 1kg  Johor Batu Pahat Hypermarket                          79   
                                                                     Kedai Runcit                         45   
                                                                     Pasar Basah                          79   
                                                                     Pasar Mini                           67   
                                                                     Pasar Raya / Supermarket             79   

                                                                                              first_week  \
project_group item_code item                   unit state district   premise_type                          
Chicken       1         AYAM BERSIH - STANDARD 1kg  Johor Batu Pahat Hypermarket              2024-12-30   
                                                                     Kedai Runcit             2024-12-30   
                                                                     Pasar Basah              2024-12-30   
                                                                     Pasar Mini               2024-12-30   
                                                                     Pasar Raya / Supermarket 2024-12-30   

                                                                                               last_week  \
project_group item_code item                   unit state district   premise_type                          
Chicken       1         AYAM BERSIH - STANDARD 1kg  Johor Batu Pahat Hypermarket              2026-06-29   
                                                                     Kedai Runcit             2026-06-29   
                                                                     Pasar Basah              2026-06-29   
                                                                     Pasar Mini               2026-06-29   
                                                                     Pasar Raya / Supermarket 2026-06-29   

                                                                                               missing_weeks_in_span  
project_group item_code item                   unit state district   premise_type                                     
Chicken       1         AYAM BERSIH - STANDARD 1kg  Johor Batu Pahat Hypermarket                                   0  
                                                                     Kedai Runcit                                 34  
                                                                     Pasar Basah                                   0  
                                                                     Pasar Mini                                   12  
                                                                     Pasar Raya / Supermarket                      0

Saved complete-scope weekly rows: 481783
Next: Notebook 05.


1743